# 5. LangChain Caching

- **LangChain Caching**은 같은 LLM 요청이 반복될 때 이전 응답을 저장해 두었다가 재사용하는 기능입니다.
- 같은 프롬프트에 대해 OpenAI에 다시 요청하지 않고 **캐시에서 꺼내** 사용합니다.
- LLM 호출 단계에서 자동으로 작동하므로, 한 번 설정하면 체인 전체에 적용됩니다.

## 학습 목표

1. 캐시가 필요한 이유와 동작 방식을 설명할 수 있습니다.
2. `InMemoryCache`로 응답 속도 차이를 확인합니다.
3. Redis Cloud를 연결해 `RedisCache`를 사용합니다.
4. 의미가 비슷한 질문도 재사용하는 **Semantic Cache**를 이해합니다.

**기준 버전**: LangChain 1.x, `langchain-redis`, 모델 `gpt-6-luna`

> **버전 변경 사항**: `langchain-community`는 유지보수가 종료되는 중(sunset)입니다. 이 노트북은 `InMemoryCache`를 `langchain_core`에서, Redis 캐시를 `langchain-redis` 패키지에서 가져옵니다.

## 1. 캐시가 필요한 이유

| 이유 | 설명 |
|---|---|
| 💰 비용 절감 | API는 호출마다 요금이 부과됩니다. 같은 요청을 반복하면 낭비가 생깁니다. |
| ⚡ 속도 향상 | 캐시된 응답은 즉시 반환됩니다. |
| 🧪 디버깅 편의 | 같은 입력을 반복 테스트할 때 매번 API를 호출하지 않아도 됩니다. |

## 2. 동작 방식

```text
캐시 미설정: Prompt → LLM → Output

캐시 설정:   Prompt → Cache 확인
                      ├─ 있으면 → 캐시 응답 반환
                      └─ 없으면 → LLM 호출 → 결과 저장 → 반환
```

LangChain 캐시의 키는 `Hash(모델 + 프롬프트 + 파라미터)`입니다.

## 3. 특징과 주의사항

- **모델 + 프롬프트 + 파라미터가 완전히 같을 때만** 캐시를 사용합니다.
- 프롬프트가 한 글자라도 다르거나, `temperature`·모델명이 다르면 다른 요청으로 처리됩니다.
- LCEL 체인 전체에 자동 적용됩니다(체인 수정 불필요).
- ⚠️ **`stream()` / `astream()` 호출은 캐시를 사용하지 않습니다.** 캐시는 `invoke()` / `batch()` 같은 일반 호출에서 동작합니다.

## 4. 캐시 종류

| 캐시 타입 | 설명 |
|---|---|
| `InMemoryCache` | 메모리(RAM)에 저장. 프로그램 종료 시 초기화(휘발성) |
| `SQLiteCache` | 로컬 SQLite 파일에 저장. 재시작해도 유지 |
| `RedisCache` | Redis 서버에 저장. 여러 서버가 공유 가능(분산 환경) |
| 커스텀 캐시 | `BaseCache`를 상속해 직접 구현 |

> 메모리 캐시의 한계: 프로그램 종료·서버 재시작 시 사라지고, 여러 서버가 공유할 수 없습니다. 실무에서는 Redis 같은 외부 캐시를 사용합니다.

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키와 모델명을 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai langchain-redis redis python-dotenv
uv add langchain-community
```

In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

## 5. InMemoryCache

데이터를 디스크나 DB가 아니라 **메모리(RAM)** 에 저장해 두고 빠르게 꺼내 쓰는 캐시입니다.

| 장점 | 설명 |
|---|---|
| 빠른 응답 | RAM에서 바로 읽으므로 디스크·DB보다 빠름 |
| 비용 절감 | 같은 요청에 API를 다시 호출하지 않음 |
| 간단한 구현 | 한 줄로 설정 |

### AI/LLM 시스템에서 캐시가 쓰이는 곳

- **LLM 응답 캐시**: 같은 프롬프트면 이전 응답을 즉시 반환 → 이 노트북의 주제
- **임베딩 캐시**: 같은 문장의 임베딩 벡터를 다시 만들지 않음
- **RAG 검색 캐시**: 자주 검색되는 질의의 검색 결과 재사용

> **KV Cache와 혼동 주의**: KV Cache는 모델 내부(GPU)에서 Transformer 추론 속도를 높이는 기술입니다. LangChain의 `InMemoryCache`는 **애플리케이션 레벨** 캐시로, 둘은 완전히 다른 계층입니다.

### 5-1. InMemoryCache 활성화

`set_llm_cache()`로 한 번 설정하면, 이후 LangChain의 모든 LLM 호출에 자동 적용됩니다.

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.prompts import ChatPromptTemplate

set_llm_cache(InMemoryCache())   # 메모리 캐시 활성화

prompt = ChatPromptTemplate.from_template("서울 남대문시장에서 가장 맛있는 길거리 음식은 무엇인가요?")
chain = prompt | llm

### 5-2. 처음 실행 (캐시 미적용) - 실행 시간 확인

In [ ]:
%%time
response = chain.invoke({})
print(response.content)

### 5-3. 같은 프롬프트로 두 번째 실행 (캐시 적용) - 실행 시간 확인

In [ ]:
%%time
response = chain.invoke({})
print(response.content)

### 5-4. 캐시 사용 / 미사용 비교

`cache=False`로 만든 모델은 전역 캐시를 쓰지 않습니다. 두 셀을 각각 두 번씩 실행해 시간을 비교해 보세요.

In [ ]:
from langchain_openai import ChatOpenAI

llm_cached = ChatOpenAI(model=default_model, reasoning_effort="none")                 # 전역 캐시 사용
llm_no_cache = ChatOpenAI(model=default_model, reasoning_effort="none", cache=False)  # 캐시 미사용

question = "서울 광장시장에서 가장 맛있는 길거리 음식은 무엇인가요?"

In [ ]:
%%time
print(llm_cached.invoke(question).content)

In [ ]:
%%time
print(llm_no_cache.invoke(question).content)

## 6. Redis 캐시

**Redis(Remote Dictionary Server)** 는 Key-Value 기반의 인메모리 데이터 저장소입니다.

- 빠른 속도를 위해 RAM에 데이터를 저장합니다.
- 캐시, 세션 저장소, 메시지 브로커 등으로 활용합니다.
- 단순 캐시가 아니라 스키마가 없는(Schema-less) **NoSQL 데이터 저장소**입니다.
- 원래는 휘발성이지만, AOF·Snapshot(RDB) 기능으로 데이터를 디스크에 보관(영속성)할 수도 있습니다.

| 구분 | RDB (MySQL 등) | Redis |
|---|---|---|
| 데이터 구조 | 테이블 기반 (스키마 필요) | Key-Value 기반 |
| 스키마 변경 | `ALTER` 필요 | 자유로운 구조 |
| 저장 위치 | 디스크 기반 | RAM 기반 (기본) |
| 속도 | 상대적으로 느림 | 매우 빠름 |

클라우드에서도 바로 사용할 수 있습니다: AWS ElastiCache, Azure Cache for Redis, **Redis Cloud** 등.

### 6-1. Redis Cloud 준비

1. [Redis Cloud](https://redis.io/try-free/)에 회원가입합니다. (무료 플랜 제공. 무료 DB는 일정 기간 사용하지 않으면 삭제될 수 있습니다.)
2. **Database 만들기**: 유료 플랜을 권하는 화면이 나오면 무료(Free, 30MB) 옵션을 선택합니다.
3. (선택) 데이터 확인용 **Redis Insight** 데스크톱 앱을 설치합니다. DB 화면의 **Connect** 버튼에서 내려받을 수 있습니다.
4. 접속 정보 확인
   - 방법 1: **Connect → Redis SDK Clients → Python** 을 선택하고 코드 예시에서 호스트·포트·비밀번호를 확인합니다.
   - 방법 2: **Databases → Configuration** 에서 Endpoint(호스트:포트)와 **Security → Default user password** 를 확인합니다.
5. ⚠️ `langchain-redis`의 캐시는 Redis의 **JSON 기능**을 사용합니다. DB를 만들 때 **Database version을 Redis 8 이상**으로 선택하세요. (Redis 7 이하 서버에서는 `unknown command 'JSON.GET'` 오류가 납니다.)
6. `.env` 파일에 아래 내용을 추가합니다.

```text
REDIS_HOST="redis-xxxxx.cloud.redislabs.com"
REDIS_PORT=10000
REDIS_PASSWORD="비밀번호"
```

In [ ]:
import os
from urllib.parse import quote_plus

import redis
from dotenv import load_dotenv
from langchain_redis import RedisChatMessageHistory

load_dotenv()
host = os.getenv("REDIS_HOST").strip()
port = int(os.getenv("REDIS_PORT", 6379))
password = os.getenv("REDIS_PASSWORD").strip()

redis_url = f"redis://default:{quote_plus(password)}@{host}:{port}"


# Redis 클라이언트: 연결 확인 및 직접 조회용
redis_client = redis.Redis.from_url(
    redis_url,
    db=0,                    # 하나의 서버에 여러 논리 DB가 있고, 0번은 기본 DB
    decode_responses=True,   # bytes 대신 문자열로 반환
)

# LangChain 대화 기록 저장소
history = RedisChatMessageHistory(
    session_id="user-1",
    redis_url=redis_url,
    ttl=3600,                # 선택: 1시간 뒤 자동 삭제 (초 단위)
)

print("Redis 연결:", redis_client.ping())

In [ ]:
import os, socket
from dotenv import load_dotenv

load_dotenv(override=True)

host = os.getenv("REDIS_HOST")
port = os.getenv("REDIS_PORT")

print(repr(host))   # 공백이나 따옴표가 있으면 여기서 보입니다
print(repr(port))

try:
    print(socket.getaddrinfo(host.strip(), int(port)))
    print("DNS 조회 성공")
except socket.gaierror as e:
    print("DNS 조회 실패:", e)

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_redis import RedisCache
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=default_model)

set_llm_cache(RedisCache(redis_client=redis_client))   # Redis 캐시 활성화

prompt = ChatPromptTemplate.from_template("서울 남대문시장에서 가장 맛있는 길거리 음식 5개를 소개해 주세요.")
chain = prompt | llm

### 6-2. 처음 실행 (캐시 미적용)

In [ ]:
%%time
response = chain.invoke({})
print(response.content)

### 6-3. 두 번째 실행 (Redis 캐시 적용)

In [ ]:
%%time
response = chain.invoke({})
print(response.content)

### 6-4. 저장된 캐시 확인

Redis Insight 앱이나 Redis Cloud 웹 화면에서 저장된 키를 확인할 수 있습니다. 파이썬으로도 확인해 봅니다.

In [ ]:
keys = redis_client.keys("redis*")   # langchain-redis 캐시 키의 기본 접두사는 "redis"
print("저장된 캐시 키 수:", len(keys))
for k in keys[:5]:
    print(k)

## 7. Semantic Cache

| 구분 | 비교 방식 | 예시 |
|---|---|---|
| 일반 캐시 (Exact Match) | 글자가 완전히 같아야 함 | "서울 날씨 어때?" ≠ "서울 날씨 어때" → 따로 API 호출 |
| 시맨틱 캐시 (Similarity Match) | **의미 유사도**로 판단 | "서울 날씨 어때?" ≈ "오늘 서울 날씨 알려줘" → 캐시 재사용 |

시맨틱 캐시는 질문을 **임베딩 벡터**로 바꿔 비슷한 질문을 찾으므로, 일반 캐시와 달리 **임베딩 모델이 반드시 필요**합니다.
(`gpt-6-luna`는 임베딩을 지원하지 않으므로 임베딩 전용 모델 `text-embedding-3-small`을 사용합니다.)

| 캐시 타입 | 특징 |
|---|---|
| `RedisSemanticCache` (`langchain-redis`) | Redis 벡터 검색 활용. 운영·분산 환경에 적합 |
| 커스텀 시맨틱 캐시 | `BaseCache`를 상속해 직접 구현. 동작 원리 학습용 |

### 7-1. 메모리 기반 시맨틱 캐시 직접 만들기

`langchain_core`의 `InMemoryVectorStore`를 사용하므로 추가 설치가 필요 없습니다.

- `lookup()`: 새 질문과 가장 비슷한 저장 질문을 찾고, **유사도가 기준값 이상**이면 저장된 응답을 반환
- `update()`: LLM 응답을 저장하고 질문을 벡터 저장소에 추가
- `similarity_threshold`: 1에 가까울수록 엄격(거의 같은 질문만 재사용). 질문 유형에 맞게 조정합니다.

In [ ]:
from typing import Any, Optional
from langchain_core.caches import BaseCache, RETURN_VAL_TYPE
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document


class SimpleSemanticCache(BaseCache):
    def __init__(self, embedding, similarity_threshold: float = 0.8):
        self.embedding = embedding
        self.similarity_threshold = similarity_threshold
        self.vectorstore = InMemoryVectorStore(embedding)
        self.store: dict[str, RETURN_VAL_TYPE] = {}

    def lookup(self, prompt: str, llm_string: str) -> Optional[RETURN_VAL_TYPE]:
        if not self.store:
            return None
        doc, score = self.vectorstore.similarity_search_with_score(prompt, k=1)[0]
        print(f"  (가장 비슷한 질문 유사도: {score:.3f})")
        if score >= self.similarity_threshold and doc.metadata["llm_string"] == llm_string:
            return self.store[doc.metadata["key"]]
        return None

    def update(self, prompt: str, llm_string: str, return_val: RETURN_VAL_TYPE) -> None:
        key = f"{prompt}::{llm_string}"
        self.store[key] = return_val
        self.vectorstore.add_documents(
            [Document(page_content=prompt, metadata={"llm_string": llm_string, "key": key})]
        )

    def clear(self, **kwargs: Any) -> None:
        self.vectorstore = InMemoryVectorStore(self.embedding)
        self.store = {}

In [ ]:
import time
from langchain_core.globals import set_llm_cache
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
set_llm_cache(SimpleSemanticCache(embedding=embeddings, similarity_threshold=0.8))

for q in ["대한민국의 수도는 어디인가요?", "한국 수도 어디야"]:
    start = time.time()
    response = llm.invoke(q)
    print(f"질문: {q}")
    print(f"답변: {response.content}")
    print(f"소요 시간: {time.time() - start:.2f}초\n")

> 두 번째 질문의 유사도가 기준값보다 낮으면 캐시를 쓰지 않고 LLM을 다시 호출합니다. 출력된 유사도를 보고 `similarity_threshold`를 조정해 보세요.
> 기준값을 너무 낮추면 **의미가 다른 질문에도 엉뚱한 캐시 답변**이 나갈 수 있으므로 주의합니다.

### 7-2. (참고) RedisSemanticCache

운영 환경에서는 Redis의 벡터 검색을 사용하는 `RedisSemanticCache`를 사용할 수 있습니다.
Redis 서버에 검색(Search & Query) 기능이 있어야 합니다. Redis Cloud에서 지원 여부를 확인한 뒤 사용하세요.

```python
from langchain_redis import RedisSemanticCache

redis_url = f"redis://default:{os.getenv('REDIS_PASSWORD')}@{os.getenv('REDIS_HOST')}:{os.getenv('REDIS_PORT')}"
set_llm_cache(RedisSemanticCache(embeddings=embeddings, redis_url=redis_url, distance_threshold=0.2))
```

> `distance_threshold`는 **거리** 기준이므로 값이 작을수록 엄격합니다. (7-1의 유사도 기준과 방향이 반대입니다.)

In [ ]:
import time
from langchain_core.globals import set_llm_cache
from langchain_openai import OpenAIEmbeddings
from langchain_redis import RedisSemanticCache

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

semantic_cache = RedisSemanticCache(
    embeddings=embeddings,
    redis_url=redis_url,        # 앞에서 만든 Redis 접속 URL 재사용
    distance_threshold=0.2,     # 거리 기준: 작을수록 엄격 (유사도 0.8 ≈ 거리 0.2)
    ttl=3600,                   # 선택: 캐시 1시간 뒤 자동 삭제
)
semantic_cache.clear()          # 실습용: 이전 실행에서 남은 캐시 제거

set_llm_cache(semantic_cache)

for q in ["대한민국의 수도는 어디인가요?", "한국 수도 어디야"]:
    start = time.time()
    response = llm.invoke(q)
    print(f"질문: {q}")
    print(f"답변: {response.content}")
    print(f"소요 시간: {time.time() - start:.2f}초\n")